Levantando as bases de dados para análise


In [39]:
import requests
import json
import pandas as pd
import plotly.express as px
from tqdm.auto import tqdm
from datetime import datetime

In [91]:
# busca via API relação de deputados em exercícío


url = 'https://dadosabertos.camara.leg.br/api/v2/deputados?ordem=ASC&ordenarPor=nome'
hd = {'accept':'application/json'}

try:
  response = requests.get(url, headers=hd)
  response.raise_for_status()

  data = response.json()

  df_deputados = pd.DataFrame(data['dados'])

except requests.exceptions.RequestException as e:
  print(f'Erro na requisição: {e}')

df_deputados.head(10)

,id,uri,nome,siglaPartido,uriPartido,siglaUf,idLegislatura,urlFoto,email
0,204379,https://dadosabertos.camara.leg.br/api/v2/depu...,Acácio Favacho,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.acaciofavacho@camara.leg.br
1,220714,https://dadosabertos.camara.leg.br/api/v2/depu...,Adail Filho,REPUBLICANOS,https://dadosabertos.camara.leg.br/api/v2/part...,AM,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adailfilho@camara.leg.br
2,221328,https://dadosabertos.camara.leg.br/api/v2/depu...,Adilson Barroso,PL,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adilsonbarroso@camara.leg.br
3,204560,https://dadosabertos.camara.leg.br/api/v2/depu...,Adolfo Viana,PSDB,https://dadosabertos.camara.leg.br/api/v2/part...,BA,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adolfoviana@camara.leg.br
4,204528,https://dadosabertos.camara.leg.br/api/v2/depu...,Adriana Ventura,NOVO,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adrianaventura@camara.leg.br
5,121948,https://dadosabertos.camara.leg.br/api/v2/depu...,Adriano do Baldy,PP,https://dadosabertos.camara.leg.br/api/v2/part...,GO,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adrianodobaldy@camara.leg.br
6,74646,https://dadosabertos.camara.leg.br/api/v2/depu...,Aécio Neves,PSDB,https://dadosabertos.camara.leg.br/api/v2/part...,MG,57,https://www.camara.leg.br/internet/deputado/ba...,dep.aecioneves@camara.leg.br
7,136811,https://dadosabertos.camara.leg.br/api/v2/depu...,Afonso Hamm,PP,https://dadosabertos.camara.leg.br/api/v2/part...,RS,57,https://www.camara.leg.br/internet/deputado/ba...,dep.afonsohamm@camara.leg.br
8,178835,https://dadosabertos.camara.leg.br/api/v2/depu...,Afonso Motta,PDT,https://dadosabertos.camara.leg.br/api/v2/part...,RS,57,https://www.camara.leg.br/internet/deputado/ba...,dep.afonsomotta@camara.leg.br
9,160527,https://dadosabertos.camara.leg.br/api/v2/depu...,Aguinaldo Ribeiro,PP,https://dadosabertos.camara.leg.br/api/v2/part...,PB,57,https://www.camara.leg.br/internet/deputado/ba...,dep.aguinaldoribeiro@camara.leg.br


Análise da tabela deputados:
id - identificação do deputado - unique 
uri - ndo deputado
nome - 
sigla partido 
sigla estado 
idLegislatura - id do período de trabalho dos parlamentares (se for só dos em exercícios será id 57)



In [93]:
df_deputados.info()

<class 'pandas.DataFrame'>
RangeIndex: 513 entries, 0 to 512
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             513 non-null    int64
 1   uri            513 non-null    str  
 2   nome           513 non-null    str  
 3   siglaPartido   513 non-null    str  
 4   uriPartido     513 non-null    str  
 5   siglaUf        513 non-null    str  
 6   idLegislatura  513 non-null    int64
 7   urlFoto        513 non-null    str  
 8   email          513 non-null    str  
dtypes: int64(2), str(7)
memory usage: 36.2 KB


In [102]:
# Cria função para chamar outras api
    
# Cria função para chamar outras api
    
def request_api(url):
    try:
        hd = {'accept': 'application/json'}
        response = requests.get(url, headers=hd)
        response.raise_for_status()
        dados = response.json()['dados']
        df = pd.json_normalize(dados)
        return df
    except requests.exceptions.RequestException as e:
        print(f'Erro na requisição do deputado {id}: {e}')
        return pd.DataFrame()
    


In [103]:
lista_dfs= []

for id in tqdm(df_deputados['id'], desc='Extract deputados detalhamento'):
   url = f'https://dadosabertos.camara.leg.br/api/v2/deputados/{id}'
   df_dados = request_api(url)
   
   if df_dados is not None and not df_dados.empty:
      lista_dfs.append(df_dados)

if lista_dfs:
   df_deputadosDetalhamento = pd.concat(lista_dfs, ignore_index=True)
else:
   df_deputadosDetalhamento = pd.DataFrame()

df_deputadosDetalhamento.head(10)

Extract deputados detalhamento: 100%|██████████| 513/513 [02:00<00:00,  4.27it/s]


,id,uri,nomeCivil,cpf,sexo,urlWebsite,redeSocial,dataNascimento,dataFalecimento,ufNascimento,...,ultimoStatus.nomeEleitoral,ultimoStatus.gabinete.nome,ultimoStatus.gabinete.predio,ultimoStatus.gabinete.sala,ultimoStatus.gabinete.andar,ultimoStatus.gabinete.telefone,ultimoStatus.gabinete.email,ultimoStatus.situacao,ultimoStatus.condicaoEleitoral,ultimoStatus.descricaoStatus
0,204379,https://dadosabertos.camara.leg.br/api/v2/depu...,ACÁCIO DA SILVA FAVACHO NETO,74287028287,M,None,"[https://twitter.com/acaciofavacho, https://ww...",1983-09-28,None,AP,...,Acácio Favacho,414,4,414,4,3215-5414,dep.acaciofavacho@camara.leg.br,Exercício,Titular,None
1,220714,https://dadosabertos.camara.leg.br/api/v2/depu...,ADAIL JOSÉ FIGUEIREDO PINHEIRO,77267796249,M,None,[https://twitter.com/adailfilhoam?s=21&t=O_eoT...,1992-02-16,None,AM,...,Adail Filho,531,4,531,5,3215-5531,dep.adailfilho@camara.leg.br,Exercício,Titular,None
2,221328,https://dadosabertos.camara.leg.br/api/v2/depu...,ADILSON BARROSO OLIVEIRA,05585378805,M,None,[],1964-06-14,None,MG,...,Adilson Barroso,603,4,603,6,3215-5603,dep.adilsonbarroso@camara.leg.br,Exercício,Efetivado,None
3,204560,https://dadosabertos.camara.leg.br/api/v2/depu...,ADOLFO VIANA DE CASTRO NETO,80123848504,M,None,[],1981-02-02,None,BA,...,Adolfo Viana,911,4,911,9,3215-5911,dep.adolfoviana@camara.leg.br,Exercício,Titular,None
4,204528,https://dadosabertos.camara.leg.br/api/v2/depu...,ADRIANA MIGUEL VENTURA,12519851813,F,None,"[https://twitter.com/adriventurasp, https://ww...",1969-03-06,None,SP,...,Adriana Ventura,802,4,802,8,3215-5802,dep.adrianaventura@camara.leg.br,Exercício,Titular,None
5,121948,https://dadosabertos.camara.leg.br/api/v2/depu...,ADRIANO ANTÔNIO AVELAR,50746553153,M,None,"[https://twitter.com/adrianodobaldy, https://w...",1969-09-06,None,GO,...,Adriano do Baldy,419,4,419,4,3215-5419,dep.adrianodobaldy@camara.leg.br,Exercício,Titular,None
6,74646,https://dadosabertos.camara.leg.br/api/v2/depu...,AÉCIO NEVES DA CUNHA,66728983791,M,None,[],1960-03-10,None,MG,...,Aécio Neves,20,x,20,None,3215-5964,dep.aecioneves@camara.leg.br,Exercício,Titular,None
7,136811,https://dadosabertos.camara.leg.br/api/v2/depu...,JOSÉ ALFONSO EBERT HAMM,37040642034,M,None,"[https://www.facebook.com/depafonsohamm, https...",1962-04-25,None,RS,...,Afonso Hamm,604,4,604,6,3215-5604,dep.afonsohamm@camara.leg.br,Exercício,Titular,None
8,178835,https://dadosabertos.camara.leg.br/api/v2/depu...,AFONSO ANTUNES DA MOTTA,10777296004,M,None,[],1950-01-08,None,RS,...,Afonso Motta,528,4,528,5,3215-5528,dep.afonsomotta@camara.leg.br,Exercício,Titular,None
9,160527,https://dadosabertos.camara.leg.br/api/v2/depu...,AGUINALDO VELLOSO BORGES RIBEIRO,51921146400,M,None,"[https://twitter.com/depaguinaldo11, https://w...",1969-02-13,None,PB,...,Aguinaldo Ribeiro,735,4,735,7,3215-5735,dep.aguinaldoribeiro@camara.leg.br,Exercício,Titular,None


In [113]:
condicaoEleitoral = df_deputadosDetalhamento['ultimoStatus.descricaoStatus'].unique()
print(condicaoEleitoral)

[None '' 'Ofício sn/2024, Dep. Célio Studart, de 27/3/2024.'
 'Ofício s/n/2023, de 17/7/2023, da Dep. Daniela do Waguinho.'
 'Ofício s/n, de 17/10/2024, Dep Danrlei de Deus Hinterholz'
 'Of. 5/25-CEDPA/P, COETICA - edição extra ao DCD de 6/5/2025 - suspensão cautelar do exercício do mandato por três meses'
 'Of. 4/GabBSB/2025, de 2/12/2025, Dep Guilherme Derrite'
 'Of. s/n/2025, de 9/4/2025, Dep. Juscelino Filho']


In [110]:
df_deputadosDetalhamento.info()

<class 'pandas.DataFrame'>
RangeIndex: 513 entries, 0 to 512
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   id                              513 non-null    int64 
 1   uri                             513 non-null    str   
 2   nomeCivil                       513 non-null    str   
 3   cpf                             513 non-null    str   
 4   sexo                            513 non-null    str   
 5   urlWebsite                      1 non-null      object
 6   redeSocial                      513 non-null    object
 7   dataNascimento                  513 non-null    str   
 8   dataFalecimento                 0 non-null      object
 9   ufNascimento                    513 non-null    str   
 10  municipioNascimento             513 non-null    str   
 11  escolaridade                    502 non-null    object
 12  ultimoStatus.id                 513 non-null    int64 
 13  u

df_DetalhamentoDeputados -  usar para deputados


 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   id                              513 non-null    int64 --- usar como chave para relação com deputados 
 1   uri                             513 non-null    str   
 2   nomeCivil                       513 non-null    str   
 3   cpf                             513 non-null    str   
 4   sexo                            513 non-null    str   
 5   urlWebsite                      1 non-null      object
 6   redeSocial                      513 non-null    object
 7   dataNascimento                  513 non-null    str   
 8   dataFalecimento                 0 non-null      object
 9   ufNascimento                    513 non-null    str   
 10  municipioNascimento             513 non-null    str   
 11  escolaridade                    502 non-null    object
 12  ultimoStatus.id                 513 non-null    int64 
 13  ultimoStatus.uri                513 non-null    str   
 14  ultimoStatus.nome               513 non-null    str   
 15  ultimoStatus.siglaPartido       513 non-null    str   
 16  ultimoStatus.uriPartido         0 non-null      object
 17  ultimoStatus.siglaUf            513 non-null    str   
 18  ultimoStatus.idLegislatura      513 non-null    int64 
 19  ultimoStatus.urlFoto            513 non-null    str   
 20  ultimoStatus.email              0 non-null      object
 21  ultimoStatus.data               513 non-null    str   
 22  ultimoStatus.nomeEleitoral      513 non-null    str   
 23  ultimoStatus.gabinete.nome      513 non-null    str   
 24  ultimoStatus.gabinete.predio    513 non-null    str   
 25  ultimoStatus.gabinete.sala      513 non-null    str   
 26  ultimoStatus.gabinete.andar     432 non-null    object
 27  ultimoStatus.gabinete.telefone  513 non-null    str   
 28  ultimoStatus.gabinete.email     513 non-null    str   
 29  ultimoStatus.situacao           513 non-null    str   
 30  ultimoStatus.condicaoEleitoral  513 non-null    str   ['Titular', 'Efetivado', 'Suplente']
 31  ultimoStatus.descricaoStatus    63 non-null     object
dtypes: int64(3), object(8), str(21)
memory usage: 128.4+ KB




In [ ]:
lista_dfs= []

for id in tqdm(df_deputados['id'], desc='Extract deputados detalhamento'):
   url = f'https://dadosabertos.camara.leg.br/api/v2/deputados/{id}'
   df_dados = request_api(url)
   
   if df_dados is not None and not df_dados.empty:
      lista_dfs.append(df_dados)

if lista_dfs:
   df_deputadosDetalhamento = pd.concat(lista_dfs, ignore_index=True)
else:
   df_deputadosDetalhamento = pd.DataFrame()

df_deputadosDetalhamento.head(10)

Extract deputados detalhamento: 100%|██████████| 513/513 [02:00<00:00,  4.27it/s]


,id,uri,nomeCivil,cpf,sexo,urlWebsite,redeSocial,dataNascimento,dataFalecimento,ufNascimento,...,ultimoStatus.nomeEleitoral,ultimoStatus.gabinete.nome,ultimoStatus.gabinete.predio,ultimoStatus.gabinete.sala,ultimoStatus.gabinete.andar,ultimoStatus.gabinete.telefone,ultimoStatus.gabinete.email,ultimoStatus.situacao,ultimoStatus.condicaoEleitoral,ultimoStatus.descricaoStatus
0,204379,https://dadosabertos.camara.leg.br/api/v2/depu...,ACÁCIO DA SILVA FAVACHO NETO,74287028287,M,None,"[https://twitter.com/acaciofavacho, https://ww...",1983-09-28,None,AP,...,Acácio Favacho,414,4,414,4,3215-5414,dep.acaciofavacho@camara.leg.br,Exercício,Titular,None
1,220714,https://dadosabertos.camara.leg.br/api/v2/depu...,ADAIL JOSÉ FIGUEIREDO PINHEIRO,77267796249,M,None,[https://twitter.com/adailfilhoam?s=21&t=O_eoT...,1992-02-16,None,AM,...,Adail Filho,531,4,531,5,3215-5531,dep.adailfilho@camara.leg.br,Exercício,Titular,None
2,221328,https://dadosabertos.camara.leg.br/api/v2/depu...,ADILSON BARROSO OLIVEIRA,05585378805,M,None,[],1964-06-14,None,MG,...,Adilson Barroso,603,4,603,6,3215-5603,dep.adilsonbarroso@camara.leg.br,Exercício,Efetivado,None
3,204560,https://dadosabertos.camara.leg.br/api/v2/depu...,ADOLFO VIANA DE CASTRO NETO,80123848504,M,None,[],1981-02-02,None,BA,...,Adolfo Viana,911,4,911,9,3215-5911,dep.adolfoviana@camara.leg.br,Exercício,Titular,None
4,204528,https://dadosabertos.camara.leg.br/api/v2/depu...,ADRIANA MIGUEL VENTURA,12519851813,F,None,"[https://twitter.com/adriventurasp, https://ww...",1969-03-06,None,SP,...,Adriana Ventura,802,4,802,8,3215-5802,dep.adrianaventura@camara.leg.br,Exercício,Titular,None
5,121948,https://dadosabertos.camara.leg.br/api/v2/depu...,ADRIANO ANTÔNIO AVELAR,50746553153,M,None,"[https://twitter.com/adrianodobaldy, https://w...",1969-09-06,None,GO,...,Adriano do Baldy,419,4,419,4,3215-5419,dep.adrianodobaldy@camara.leg.br,Exercício,Titular,None
6,74646,https://dadosabertos.camara.leg.br/api/v2/depu...,AÉCIO NEVES DA CUNHA,66728983791,M,None,[],1960-03-10,None,MG,...,Aécio Neves,20,x,20,None,3215-5964,dep.aecioneves@camara.leg.br,Exercício,Titular,None
7,136811,https://dadosabertos.camara.leg.br/api/v2/depu...,JOSÉ ALFONSO EBERT HAMM,37040642034,M,None,"[https://www.facebook.com/depafonsohamm, https...",1962-04-25,None,RS,...,Afonso Hamm,604,4,604,6,3215-5604,dep.afonsohamm@camara.leg.br,Exercício,Titular,None
8,178835,https://dadosabertos.camara.leg.br/api/v2/depu...,AFONSO ANTUNES DA MOTTA,10777296004,M,None,[],1950-01-08,None,RS,...,Afonso Motta,528,4,528,5,3215-5528,dep.afonsomotta@camara.leg.br,Exercício,Titular,None
9,160527,https://dadosabertos.camara.leg.br/api/v2/depu...,AGUINALDO VELLOSO BORGES RIBEIRO,51921146400,M,None,"[https://twitter.com/depaguinaldo11, https://w...",1969-02-13,None,PB,...,Aguinaldo Ribeiro,735,4,735,7,3215-5735,dep.aguinaldoribeiro@camara.leg.br,Exercício,Titular,None


In [105]:
# Extração de dados legislaturas

url = f'https://dadosabertos.camara.leg.br/api/v2/legislaturas'
df_Legistaluras = request_api(url)

df_Legistaluras.head()

,id,uri,dataInicio,dataFim
0,57,https://dadosabertos.camara.leg.br/api/v2/legi...,2023-02-01,2027-01-31
1,56,https://dadosabertos.camara.leg.br/api/v2/legi...,2019-02-01,2023-01-31
2,55,https://dadosabertos.camara.leg.br/api/v2/legi...,2015-02-01,2019-01-31
3,54,https://dadosabertos.camara.leg.br/api/v2/legi...,2011-02-01,2015-01-31
4,53,https://dadosabertos.camara.leg.br/api/v2/legi...,2007-02-01,2011-01-31


In [23]:
# Extração dados das frentes parlamentares
lista_dfs = []

for id in tqdm(df_deputados['id'], desc='Extract Deputados'):
    # Coleta os dados do deputado atual
    url = f'https://dadosabertos.camara.leg.br/api/v2/deputados/{id}/frentes'
    df_dados = request_api(url)
    
    # 2. Verifica se o retorno não é vazio antes de adicionar à lista
    if df_dados is not None and not df_dados.empty:
        lista_dfs.append(df_dados)

# 3. Concatena todos os DataFrames da lista de uma só vez
if lista_dfs:
    df_Frente = pd.concat(lista_dfs, ignore_index=True)
else:
    df_Frente = pd.DataFrame() # Cria um DF vazio caso não encontre dados

Processando Deputados: 100%|██████████| 513/513 [02:04<00:00,  4.12it/s]


In [115]:
url = f'https://dadosabertos.camara.leg.br/api/v2/partidos'
df_Partidos = request_api(url)

In [116]:
df_Partidos.head(5)

,id,sigla,nome,uri
0,36898,AVANTE,Avante,https://dadosabertos.camara.leg.br/api/v2/part...
1,37905,CIDADANIA,Cidadania,https://dadosabertos.camara.leg.br/api/v2/part...
2,36899,MDB,Movimento Democrático Brasileiro,https://dadosabertos.camara.leg.br/api/v2/part...
3,38011,MISSÃO,Partido Missão,https://dadosabertos.camara.leg.br/api/v2/part...
4,37901,NOVO,Partido Novo,https://dadosabertos.camara.leg.br/api/v2/part...


In [118]:
lista_dfs = []

for id in tqdm(df_Partidos['id'], desc= 'extrect partidos detalhamento'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/partidos/{id}'
    df_dados = request_api(url)

    # 2. Verifica se o retorno não é vazio antes de adicionar à lista
    if df_dados is not None and not df_dados.empty:
        lista_dfs.append(df_dados)

# 3. Concatena todos os DataFrames da lista de uma só vez
if lista_dfs:
    df_PartidosDetalhado = pd.concat(lista_dfs, ignore_index=True)
else:
    df_PartidosDetalhado = pd.DataFrame() # Cria um DF vazio caso não encontre dados    

extrect partidos detalhamento:   0%|          | 0/15 [00:00<?, ?it/s]

extrect partidos detalhamento: 100%|██████████| 15/15 [00:03<00:00,  3.98it/s]


In [119]:
df_PartidosDetalhado.head(5)

,id,sigla,nome,uri,numeroEleitoral,urlLogo,urlWebSite,urlFacebook,status.data,status.idLegislatura,...,status.totalPosse,status.totalMembros,status.uriMembros,status.lider.uri,status.lider.nome,status.lider.siglaPartido,status.lider.uriPartido,status.lider.uf,status.lider.idLegislatura,status.lider.urlFoto
0,36898,AVANTE,Avante,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2026-02-19T15:27,57,...,7,8,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Bruno Farias,AVANTE,https://dadosabertos.camara.leg.br/api/v2/part...,MG,57,https://www.camara.leg.br/internet/deputado/ba...
1,37905,CIDADANIA,Cidadania,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2021-03-02T13:23,57,...,5,4,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Alex Manente,CIDADANIA,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...
2,36899,MDB,Movimento Democrático Brasileiro,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2024-10-07T08:39,57,...,41,43,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Isnaldo Bulhões Jr.,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AL,57,https://www.camara.leg.br/internet/deputado/ba...
3,38011,MISSÃO,Partido Missão,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2026-03-10T12:14,57,...,0,1,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Kim Kataguiri,MISSÃO,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...
4,37901,NOVO,Partido Novo,https://dadosabertos.camara.leg.br/api/v2/part...,None,https://www.camara.leg.br/internet/Deputado/im...,None,None,2025-04-24T19:20,57,...,3,5,https://dadosabertos.camara.leg.br/api/v2/depu...,https://dadosabertos.camara.leg.br/api/v2/depu...,Marcel van Hattem,NOVO,https://dadosabertos.camara.leg.br/api/v2/part...,RS,57,https://www.camara.leg.br/internet/deputado/ba...


In [10]:
df_Frente.info()

<class 'pandas.DataFrame'>
RangeIndex: 130939 entries, 0 to 130938
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             130939 non-null  int64
 1   uri            130939 non-null  str  
 2   titulo         130939 non-null  str  
 3   idLegislatura  130939 non-null  int64
 4   id_deputado    130939 non-null  int64
 5   fonte          130939 non-null  str  
dtypes: int64(3), str(3)
memory usage: 6.0 MB


In [48]:
# Extração de proposições
hoje =datetime.now()
lista_dfs = []
for ano in tqdm(range(2022,hoje.year), desc='Extract proposições'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes?ano={ano}&ordem=ASC&ordenarPor=id'
    df_dados = request_api(url)

    if df_dados is not None and not df_dados.empty:
        lista_dfs.append(df_dados)

if lista_dfs:
    df_Proposicoes = pd.concat(lista_dfs, ignore_index=True)

else:
    df_Proposicoes = pd.DataFrame()

Extract proposições: 100%|██████████| 4/4 [00:07<00:00,  1.75s/it]


In [45]:
df_Proposicoes.head()

,id,uri,siglaTipo,codTipo,numero,ano,ementa,dataApresentacao
0,303153,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,618,2022,Dispõe sobre o exercício da profissão de Podól...,2005-10-11T14:41
1,493361,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,2253,2022,Dispõe sobre o monitoramento por instrumentos ...,2011-02-23T19:42
2,587207,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1367,2022,Dispõe sobre a prestação dos serviços de contr...,2013-08-14T18:06
3,597587,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,3062,2022,"Altera a redação dos arts. 14, 17 e 18 da Lei ...",2013-10-22T10:53
4,996806,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1637,2022,Dispõe sobre a avaliação psicológica de gestan...,2015-03-12T15:02


In [46]:
df_Proposicoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id                60 non-null     int64
 1   uri               60 non-null     str  
 2   siglaTipo         60 non-null     str  
 3   codTipo           60 non-null     int64
 4   numero            60 non-null     int64
 5   ano               60 non-null     int64
 6   ementa            60 non-null     str  
 7   dataApresentacao  60 non-null     str  
dtypes: int64(4), str(4)
memory usage: 3.9 KB


In [49]:
#Extração autores da s proposições

lista_dfs = []

for p in tqdm(df_Proposicoes['id'], desc= 'Extract autores das proposições'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{p}/autores'
    df_dados = request_api(url)

    if df_dados is not None and not df_dados.empty:
        lista_dfs.append(df_dados)

    if lista_dfs:
        df_AutoresProposicoes = pd.concat(lista_dfs, ignore_index=True)
    else:
        df_AutoresProposicoes = pd.DataFrame()

Extract autores das proposições: 100%|██████████| 60/60 [00:16<00:00,  3.57it/s]


In [50]:
df_AutoresProposicoes.head()

,uri,nome,codTipo,tipo,ordemAssinatura,proponente
0,https://dadosabertos.camara.leg.br/api/v2/depu...,José Mentor,10000,Deputado(a),1,1
1,https://dadosabertos.camara.leg.br/api/v2/depu...,Pedro Paulo,10000,Deputado(a),1,1
2,https://dadosabertos.camara.leg.br/api/v2/depu...,Laercio Oliveira,10000,Deputado(a),1,1
3,https://dadosabertos.camara.leg.br/api/v2/depu...,Ricardo Izar,10000,Deputado(a),1,1
4,https://dadosabertos.camara.leg.br/api/v2/depu...,Célio Silveira,10000,Deputado(a),1,1


In [51]:
# Extração tipos de proposição

url = 'https://dadosabertos.camara.leg.br/api/v2/referencias/tiposProposicao'
df_TiposProposicao = request_api(url)

In [52]:
df_TiposProposicao.head()

,cod,sigla,nome,descricao
0,27,OF,Ofício do Congresso Nacional,
1,129,CON,Consulta,Consulta
2,130,EMC,Emenda na Comissão,Emenda Apresentada na Comissão
3,131,EMP,Emenda de Plenário,Emenda de Plenário
4,132,EMS,Emenda/Substitutivo do Senado,Emenda/Substitutivo do Senado


In [53]:
# estração de temas das proposições 

lista_dfs = []

for p in tqdm(df_Proposicoes['id'], desc= 'Extract autores das proposições'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{p}/autores'
    df_dados = request_api(url)

    if df_dados is not None and not df_dados.empty:
        lista_dfs.append(df_dados)

    if lista_dfs:
        df_TemasProposicoes = pd.concat(lista_dfs, ignore_index=True)
    else:
        df_TemasProposicoes = pd.DataFrame()

Extract autores das proposições: 100%|██████████| 60/60 [00:12<00:00,  4.88it/s]


In [54]:
df_TemasProposicoes.head()

,uri,nome,codTipo,tipo,ordemAssinatura,proponente
0,https://dadosabertos.camara.leg.br/api/v2/depu...,José Mentor,10000,Deputado(a),1,1
1,https://dadosabertos.camara.leg.br/api/v2/depu...,Pedro Paulo,10000,Deputado(a),1,1
2,https://dadosabertos.camara.leg.br/api/v2/depu...,Laercio Oliveira,10000,Deputado(a),1,1
3,https://dadosabertos.camara.leg.br/api/v2/depu...,Ricardo Izar,10000,Deputado(a),1,1
4,https://dadosabertos.camara.leg.br/api/v2/depu...,Célio Silveira,10000,Deputado(a),1,1


In [64]:
# estração de trramitação das proposições 

lista_dfs = []

for p in tqdm(df_Proposicoes['id'], desc= 'Extract tramitação das proposicoes'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{p}/tramitacoes'
    df_dados = request_api(url)

    if df_dados is not None and not df_dados.empty:
        lista_dfs.append(df_dados)

    if lista_dfs:
        df_Tramitacao = pd.concat(lista_dfs, ignore_index=True)
    else:
        df_Tramitacao = pd.DataFrame()

Extract tramitação das proposicoes: 100%|██████████| 60/60 [00:19<00:00,  3.05it/s]


In [65]:
df_Tramitacao.head()

,dataHora,sequencia,siglaOrgao,uriOrgao,uriUltimoRelator,regime,descricaoTramitacao,codTipoTramitacao,descricaoSituacao,codSituacao,despacho,url,ambito,apreciacao
0,2005-10-11T14:41,1,PLEN,https://dadosabertos.camara.leg.br/api/v2/orga...,NaN,"Ordinário (Art. 151, III, RICD)",Apresentação de Proposição,100,Pronta para Pauta,924.0,Apresentação do Projeto de Lei pelo Deputado J...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
1,2005-10-20T15:28,10,MESA,https://dadosabertos.camara.leg.br/api/v2/orga...,NaN,"Ordinário (Art. 151, III, RICD)",Distribuição,110,Aguardando análise de prazo recursal,1050.0,Às Comissões de Seguridade Social e Família; T...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
2,2005-10-25T00:00,15,CCP,https://dadosabertos.camara.leg.br/api/v2/orga...,NaN,"Ordinário (Art. 151, III, RICD)",Publicação de Proposição,604,Aguardando Encaminhamento,910.0,Encaminhada à publicação. Publicação Inicial n...,https://www.camara.leg.br/proposicoesWeb/prop_...,Regimental,Proposição Sujeita à Apreciação do Plenário
3,2005-12-01T00:00,17,CSAUDE,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Ordinário (Art. 151, III, RICD)",Designação de Relator(a),320,Aguardando Encaminhamento,910.0,"Designada Relatora, Dep. Almerinda de Carvalho...",NaN,Regimental,Proposição Sujeita à Apreciação do Plenário
4,2005-12-02T00:00,18,CSAUDE,https://dadosabertos.camara.leg.br/api/v2/orga...,https://dadosabertos.camara.leg.br/api/v2/depu...,"Ordinário (Art. 151, III, RICD)",Abertura de Prazo,350,Aguardando Encaminhamento,910.0,Prazo para Emendas ao Projeto (5 sessões ordin...,NaN,Regimental,Proposição Sujeita à Apreciação do Plenário


In [69]:
df_TiposTramitacao = df_Tramitacao['descricaoTramitacao'].drop_duplicates

df_TiposProposicao.info()

<class 'pandas.DataFrame'>
RangeIndex: 544 entries, 0 to 543
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   cod        544 non-null    str  
 1   sigla      544 non-null    str  
 2   nome       544 non-null    str  
 3   descricao  541 non-null    str  
dtypes: str(4)
memory usage: 17.1 KB


In [70]:
# extracao de votações

lista_dfs = []

for p in tqdm(df_Proposicoes['id'], desc= 'Extract autores das proposições'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes/{p}/votacoes'
    df_dados = request_api(url)

    if df_dados is not None and not df_dados.empty:
        lista_dfs.append(df_dados)

    if lista_dfs:
        df_Votacoes = pd.concat(lista_dfs, ignore_index=True)
    else:
        df_Votacoes = pd.DataFrame()

Extract autores das proposições: 100%|██████████| 60/60 [02:39<00:00,  2.66s/it]


In [72]:
# esxtração cod Tipo de Tramitação

url=  'https://dadosabertos.camara.leg.br/api/v2/referencias/proposicoes/codTipoTramitacao'
df_CodTiposTramitacao = request_api(url)

df_CodTiposTramitacao.head()

,cod,sigla,nome,descricao
0,5,,Não Informado,
1,100,,Apresentação de Proposição,
2,104,,Desapensação,
3,105,,Leitura e publicação,
4,106,,Apensação,


In [73]:
# Extração de votacoes
hoje =datetime.now()
lista_dfs = []
for ano in tqdm(range(2022,hoje.year), desc='Extract votações'):
    url = f'https://dadosabertos.camara.leg.br/api/v2/proposicoes?ano={ano}&ordem=ASC&ordenarPor=id'
    df_dados = request_api(url)

    if df_dados is not None and not df_dados.empty:
        lista_dfs.append(df_dados)

if lista_dfs:
    df_VotacoesTotais = pd.concat(lista_dfs, ignore_index=True)

else:
    df_VotacoesTotais = pd.DataFrame()

Extract votações: 100%|██████████| 4/4 [00:09<00:00,  2.34s/it]


In [74]:
df_VotacoesTotais.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id                60 non-null     int64
 1   uri               60 non-null     str  
 2   siglaTipo         60 non-null     str  
 3   codTipo           60 non-null     int64
 4   numero            60 non-null     int64
 5   ano               60 non-null     int64
 6   ementa            60 non-null     str  
 7   dataApresentacao  60 non-null     str  
dtypes: int64(4), str(4)
memory usage: 3.9 KB


In [76]:
df_VotacoesTotais.head()

,id,uri,siglaTipo,codTipo,numero,ano,ementa,dataApresentacao
0,303153,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,618,2022,Dispõe sobre o exercício da profissão de Podól...,2005-10-11T14:41
1,493361,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,2253,2022,Dispõe sobre o monitoramento por instrumentos ...,2011-02-23T19:42
2,587207,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1367,2022,Dispõe sobre a prestação dos serviços de contr...,2013-08-14T18:06
3,597587,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,3062,2022,"Altera a redação dos arts. 14, 17 e 18 da Lei ...",2013-10-22T10:53
4,996806,https://dadosabertos.camara.leg.br/api/v2/prop...,PL,139,1637,2022,Dispõe sobre a avaliação psicológica de gestan...,2015-03-12T15:02
